In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
from tqdm import tqdm
import dask.dataframe as dd
from dask.diagnostics import ProgressBar

In [2]:
# Check if a GPU is available; otherwise use CPU
device = 'cuda:2' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {device}')

In [3]:
learn_w = 4
lead_w = 0
pred_w = 4

test_ratio = 0.2                 # Test set ratio
valid_ratio = 0.1                # Validation set ratio

input_dim = 41
hidden_dim = 64
num_layers = 6
output_dim = 2

seed = 42                        # Random seed
batch_size = 64                  # Batch size
num_epoch = 300                  # Number of training epochs
learning_rate = 0.00001          # Learning rate

pre_model_path = './eicu_mimic_pre_dead_rate.ckpt'
model_path = './lstm_all_params_best.ckpt'     # Path to save checkpoints (used by the model-saving function below)


In [4]:
def same_seeds(seed):  # Fix random seed (CPU)
    torch.manual_seed(seed)  # Fix random seed (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)  # Set seed for the current GPU
        torch.cuda.manual_seed_all(seed)  # Set seed for all GPUs
    np.random.seed(seed)  # Ensure deterministic random numbers when using numpy/random later
    torch.backends.cudnn.benchmark = False  # If GPU and network structure are fixed, you can set this to True
    torch.backends.cudnn.deterministic = True  # Make operations deterministic (fixed network behavior)

same_seeds(seed)


In [5]:

sepsis_mimic_train_data = pd.read_csv('../sepsis_mimic_train_data.csv')
sepsis_mimic_test_data = pd.read_csv('../sepsis_mimic_test_data.csv')
sepsis_mimic_valid_data = pd.read_csv('../sepsis_mimic_valid_data.csv')

sepsis_eicu_data = pd.read_csv('../sepsis_eicu_data.csv')

In [6]:
class SepsisData(Dataset):
    def __init__(self, data, learn_w, lead_w, pred_w):
        x1s = []#Storing non-invasive data
        x2s = []#Storing invasive data
        ys = []
        stayid = []
        row_id = list(set(data['stay_id']))
        for i in row_id:
            data_row = data[data['stay_id'] == i]#Retrieve data for each patient ID
            data_row = data_row.sort_values(by = 'hr')#Sort by time
            data_row = data_row.reset_index(drop=True)
            end_boundary = data_row.shape[0]#When there is no positive data, the boundary is the entire table.
            for j in range(data_row.shape[0]):
                if data_row.iloc[j, -1] == 1:#The row with a positive boundary
                    end_boundary = j + 1
                    break

            if (end_boundary <= learn_w):#If the boundary is smaller than the learning window, the patient will not be considered.
                continue
            
            for j in range(end_boundary - learn_w - lead_w):
                label = []
                x_data = data_row.iloc[j: j + learn_w, 1:-1]#Data in the learning window
                x1_data = x_data.loc[:, ['heart_rate', 'mbp', 'temperature', 'spo2', 'resp_rate', 'sbp', 'dbp']]
                x2_data = x_data.drop(['heart_rate', 'mbp', 'temperature', 'spo2', 'resp_rate', 'sbp', 'dbp'], axis =1).iloc[-1, :]
                positive_flag_6 = 0
                for k in range(j + learn_w + lead_w, min(j + lead_w + learn_w + 6, data_row.shape[0])):# Label the data within the prediction window as positive.
                    if (data_row.iloc[k, -1] == 1):
                        positive_flag_6 = 1
                        label.append(1)
                        break
                if (positive_flag_6 == 0):
                    label.append(0)
                positive_flag_12 = 0
                for k in range(j + learn_w + lead_w, min(j + lead_w + learn_w + 12, data_row.shape[0])):
                    if (data_row.iloc[k, -1] == 1):
                        positive_flag_12 = 1
                        label.append(1)
                        break
                if (positive_flag_12 == 0):
                    label.append(0)
                positive_flag_24 = 0
                for k in range(j + learn_w + lead_w, min(j + lead_w + learn_w + 24, data_row.shape[0])):
                    if (data_row.iloc[k, -1] == 1):
                        positive_flag_24 = 1
                        label.append(1)
                        break
                if (positive_flag_24 == 0):
                    label.append(0)

                x1s.append(torch.tensor(x1_data.values, dtype=torch.float32))
                x2s.append(torch.tensor(x2_data.values, dtype=torch.float32))
                ys.append(torch.tensor(label, dtype=torch.float32))
                stayid.append(torch.tensor(i, dtype=torch.float32))

        self.x1 = torch.stack(x1s)
        self.x2 = torch.stack(x2s)
        self.y = torch.stack(ys)
        self.sid = torch.stack(stayid)
        
    
    def __getitem__(self, index):
        return self.x1[index], self.x2[index],self.y[index], self.sid[index]
    
    def __len__(self):
        return len(self.x1)

In [ ]:
train_set = SepsisData(sepsis_mimic_train_data, learn_w, lead_w, pred_w)
test_set = SepsisData(sepsis_mimic_test_data, learn_w, lead_w, pred_w)
valid_set = SepsisData(sepsis_mimic_valid_data, learn_w, lead_w, pred_w)
eicu_set = SepsisData(sepsis_eicu_data, learn_w, lead_w, pred_w)

In [23]:
train_loader = DataLoader(train_set, batch_size=4096, shuffle=True, num_workers=16, pin_memory=True, prefetch_factor=2)
test_loader = DataLoader(test_set, batch_size=4096, shuffle=True, num_workers=16, pin_memory=True, prefetch_factor=2)
valid_loader = DataLoader(valid_set, batch_size=4096, shuffle=True, num_workers=16, pin_memory=True, prefetch_factor=2)
eicu_loader = DataLoader(eicu_set, batch_size=4096, shuffle=True, num_workers=16, pin_memory=True, prefetch_factor=2)

In [9]:
class BasicBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(BasicBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU(),
            nn.Dropout(p=0.3)
        )
    def forward(self, x):
        x = self.block(x)
        return x
    
class GRU(nn.Module):
    def __init__(self, x1_input_dim, x1_hidden_dim, x1_num_layers, x2_input_dim, x2_hidden_dim, x2_num_layers, hidden_dim, num_layers, output_dim):
        super(GRU, self).__init__()
        self.x1_hidden_dim = x1_hidden_dim
        self.x1_num_layers = x1_num_layers
        self.x1_fc_input = nn.Sequential(
            nn.Linear(x1_input_dim, x1_hidden_dim),
            nn.LayerNorm(x1_hidden_dim),
            nn.Sigmoid(),
            nn.Dropout(0.3)
        ) 
        self.gru = nn.GRU(x1_hidden_dim, x1_hidden_dim, x1_num_layers, batch_first=True, dropout=0.5)

        self.gln = nn.Sequential(
            nn.LayerNorm(x1_hidden_dim)
        )
        self.x1_fc_output = nn.Linear(x1_hidden_dim, x1_hidden_dim)
        self.x2_fc = nn.Sequential(
            BasicBlock(x2_input_dim, x2_hidden_dim),
            *[BasicBlock(x2_hidden_dim, x2_hidden_dim) for _ in range(x2_num_layers -1)],
        )
        self.fc = nn.Sequential(
            BasicBlock(x1_hidden_dim + x2_hidden_dim, hidden_dim),
            *[BasicBlock(hidden_dim, hidden_dim) for _ in range(num_layers -1)],
        )
        self.last_fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x1, x2):#x1 Non-invasive, x2 Invasive
        out_x1 = self.x1_fc_input(x1)
        h0 = torch.zeros(self.x1_num_layers, out_x1.size(0), self.x1_hidden_dim).requires_grad_().to(device)
        out_x1, hn = self.gru(out_x1, h0.detach())
        out_x1 = self.gln(out_x1)
        out_x1 = self.x1_fc_output(out_x1[:, -1, :])
        out_x2 = self.x2_fc(x2)
        out = torch.cat((out_x1, out_x2), dim=1)
        out = self.fc(out)
        out = self.last_fc(out)
        return out

In [11]:
pre_model = GRU(x1_input_dim=7, x1_hidden_dim=32, x1_num_layers=8, x2_input_dim=34, x2_hidden_dim=64, x2_num_layers=10, hidden_dim=64, num_layers=8, output_dim=2).to(device)
pre_model.load_state_dict(torch.load(pre_model_path))
print(pre_model.modules)

In [12]:
class BasicBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(BasicBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU(),
            nn.Dropout(p=0.3)
        )
    def forward(self, x):
        x = self.block(x)
        return x
class MyModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers = 1):
        super(MyModel, self).__init__()
        self.pre_x1_fc_input = pre_model.x1_fc_input
        self.pre_gru = pre_model.gru
        self.pre_gln = pre_model.gln
        self.pre_x1_fc_output = pre_model.x1_fc_output
        self.pre_x2_fc = pre_model.x2_fc
        num_features = pre_model.x1_hidden_dim + 64
        self.pre_fc = pre_model.fc
        self.fc = nn.Sequential(
            BasicBlock(64, hidden_dim),
            *[BasicBlock(hidden_dim, hidden_dim) for _ in range(num_layers - 1)],
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x1, x2):
        out_x1 = self.pre_x1_fc_input(x1)
        h0 = torch.zeros(pre_model.x1_num_layers, out_x1.size(0), pre_model.x1_hidden_dim).requires_grad_().to(device)
        out_x1, hn = self.pre_gru(out_x1, h0.detach())
        out_x1 = self.pre_gln(out_x1)
        out_x1 = self.pre_x1_fc_output(out_x1[:, -1, :])
        out_x2 = self.pre_x2_fc(x2)
        out = torch.cat((out_x1, out_x2), dim=1)
        out = self.pre_fc(out)
        out = self.fc(out)
        return out

In [13]:
model = MyModel(hidden_dim=32, output_dim=3, num_layers=4).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-3)
print(model.children)

In [ ]:
from sklearn.metrics import roc_auc_score
best_acc = 0.0
best_auc = 0.0
stale = 0
patience = 30
for epoch in range(num_epoch):
    train_acc = 0.0
    train_loss = 0.0
    val_acc = 0.0
    val_loss = 0.0

    model.train()
    train_loss = []
    train_accs = []
    train_aucs = []
    
    train_prob_all_6 = []
    train_label_all_6 = []
    train_prob_all_12 = []
    train_label_all_12 = []
    train_prob_all_24 = []
    train_label_all_24 = []
    for batch in tqdm(train_loader):
        x1, x2, labels = batch
        x1 = x1.to(device)
        x2 = x2.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(x1, x2)
 
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss.append(loss.item())

        train_prob_all_6.extend(outputs[:, 0].detach().cpu().numpy())
        train_label_all_6.extend(labels[:, 0].detach().cpu().numpy())
        train_prob_all_12.extend(outputs[:, 1].detach().cpu().numpy())
        train_label_all_12.extend(labels[:, 1].detach().cpu().numpy())
        train_prob_all_24.extend(outputs[:, 2].detach().cpu().numpy())
        train_label_all_24.extend(labels[:, 2].detach().cpu().numpy())
        


    mean_train_loss = sum(train_loss)/len(train_loss)
    train_auc_6 = roc_auc_score(train_label_all_6, train_prob_all_6)
    train_auc_12 = roc_auc_score(train_label_all_12, train_prob_all_12)
    train_auc_24 = roc_auc_score(train_label_all_24, train_prob_all_24)
    print(f"[ Train | {epoch + 1:03d}/{num_epoch:03d} ] loss = {mean_train_loss:.5f}, auc = {((train_auc_6 + train_auc_12 + train_auc_24) / 3):.5f}")
    
    
    valid_loss = []
    valid_accs = []
    valid_aucs = []
    valid_prob_all_6 = []
    valid_label_all_6 = []
    valid_prob_all_12 = []
    valid_label_all_12 = []
    valid_prob_all_24 = []
    valid_label_all_24 = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(train_loader):
            x1, x2, labels = batch
            x1 = x1.to(device)
            x2 = x2.to(device)
            labels = labels.to(device)
            outputs = model(x1, x2)

            loss = criterion(outputs, labels)
            valid_loss.append(loss.item())

            valid_prob_all_6.extend(outputs[:, 0].detach().cpu().numpy())
            valid_label_all_6.extend(labels[:, 0].detach().cpu().numpy())
            valid_prob_all_12.extend(outputs[:, 1].detach().cpu().numpy())
            valid_label_all_12.extend(labels[:, 1].detach().cpu().numpy())
            valid_prob_all_24.extend(outputs[:, 2].detach().cpu().numpy())
            valid_label_all_24.extend(labels[:, 2].detach().cpu().numpy())
    
    mean_valid_loss = sum(valid_loss) / len(valid_loss)
    valid_auc_6 = roc_auc_score(valid_label_all_6, valid_prob_all_6)
    valid_auc_12 = roc_auc_score(valid_label_all_12, valid_prob_all_12)
    valid_auc_24 = roc_auc_score(valid_label_all_24, valid_prob_all_24)
    mean_valid_auc = ((valid_auc_6 + valid_auc_12 + valid_auc_24) / 3)
    print(f"[ Valid | {epoch + 1:03d}/{num_epoch:03d} ] loss = {mean_valid_loss:.5f}, auc = {mean_valid_auc:.5f}")
    # save models
    if mean_valid_auc > best_auc:
        print(f"Best model found at epoch {epoch}, saving model")
        torch.save(model.state_dict(), model_path) # only save best to prevent output memory exceed error
        best_auc = mean_valid_auc
        stale = 0
    else:
        stale += 1
        if stale > patience:
            print(f"No improvment {patience} consecutive epochs, early stopping")
            break


In [24]:
model = MyModel(hidden_dim=32, output_dim=3, num_layers=4).to(device)
model.load_state_dict(torch.load(model_path))

In [40]:
from sklearn.metrics import roc_auc_score, confusion_matrix
torch.backends.cudnn.enabled = True
model.eval()
test_acc = 0.0
threshold = 0.2
test_prob_all_6 = []
test_label_all_6 = []
test_prob_all_12 = []
test_label_all_12 = []
test_prob_all_24 = []
test_label_all_24 = []
result_id = []
with torch.no_grad():
    for i, batch in enumerate(tqdm(train_loader)):
        x1, x2, labels, sid = batch
        x1 = x1.to(device)
        x2 = x2.to(device)
        labels = labels.to(device)
        outputs = model(x1, x2)
        outputs = torch.sigmoid(outputs)
        
        test_prob_all_6.extend(outputs[:, 0].detach().cpu().numpy())
        test_label_all_6.extend(labels[:, 0].detach().cpu().numpy())
        test_prob_all_12.extend(outputs[:, 1].detach().cpu().numpy())
        test_label_all_12.extend(labels[:, 1].detach().cpu().numpy())
        test_prob_all_24.extend(outputs[:, 2].detach().cpu().numpy())
        test_label_all_24.extend(labels[:, 2].detach().cpu().numpy())
        result_id.extend(sid)
test_prob_all_6 = np.array(test_prob_all_6)
test_label_all_6 = np.array(test_label_all_6)
test_prob_all_12 = np.array(test_prob_all_12)
test_label_all_12 = np.array(test_label_all_12)
test_prob_all_24 = np.array(test_prob_all_24)
test_label_all_24 = np.array(test_label_all_24)
test_pred_all = (test_prob_all_12 >= threshold).astype(int)
test_auc_6 = roc_auc_score(test_label_all_6, test_prob_all_6)
test_auc_12 = roc_auc_score(test_label_all_12, test_prob_all_12)
test_auc_24 = roc_auc_score(test_label_all_24, test_prob_all_24)
mean_auc = (test_auc_6 + test_auc_12 + test_auc_24) / 3
print(f"acc = {(test_label_all_6 == test_pred_all).sum() / len(test_pred_all):.5f}, auc = {mean_auc:.5f}")

cm = confusion_matrix(test_label_all_12, test_pred_all)
TN, FP, FN, TP = cm.ravel()

# 计算 TPR 和 TNR   
TPR = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # Sensitivity / Recall
TNR = TN / (TN + FP) if (TN + FP) > 0 else 0.0  # Specificity
print(f"Confusion Matrix:\n{cm}")
print(f"TPR (Sensitivity): {TPR:.4f}")
print(f"TNR (Specificity): {TNR:.4f}")

## Modified confidence interval calculation method

In [41]:
import os
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed


def _clean(pid, y, p):
    pid = np.asarray(pid)
    y = np.asarray(y).astype(np.int8)

    p = np.asarray(p).astype(np.float64)
    mask = np.isfinite(p) & np.isfinite(y) & (pid != None)
    pid, y, p = pid[mask], y[mask], p[mask]

    if np.unique(y).size < 2:
        raise ValueError("There is only one category in the entire dataset, so it cannot be calculated.")

    # 千分位整数：0~1000
    p_int = np.rint(p * 1000.0).astype(np.int16)
    p_int = np.clip(p_int, 0, 1000)

    return pid, y, p_int


def _group_prepare(pid, y, p):

    pid, y, p_int = _clean(pid, y, p)

    unique_pids, pid_code = np.unique(pid, return_inverse=True)
    N = unique_pids.size

    order = np.argsort(p_int, kind="mergesort")
    p_s = p_int[order]
    y_s = y[order].astype(np.int8)
    pid_code_s = pid_code[order]

    scores_int, group_id = np.unique(p_s, return_inverse=True)
    return N, pid_code_s, y_s, scores_int, group_id

def _weighted_auc_from_group(pos_w, neg_w):
    W_pos = pos_w.sum()
    W_neg = neg_w.sum()
    if W_pos <= 0 or W_neg <= 0:
        return None

    cum_neg_before = np.concatenate(([0.0], np.cumsum(neg_w)[:-1]))
    num = np.sum(pos_w * (cum_neg_before + 0.5 * neg_w))
    return float(num / (W_pos * W_neg))

def _threshold_scan_grid_from_group_int(scores_int_asc, pos_w_asc, neg_w_asc,
                                        thr_start=0.10, thr_stop=0.30, thr_step=0.01):

    W_pos = pos_w_asc.sum()
    W_neg = neg_w_asc.sum()
    if W_pos <= 0 or W_neg <= 0:
        return None

    pos_ge = np.cumsum(pos_w_asc[::-1])[::-1]
    neg_ge = np.cumsum(neg_w_asc[::-1])[::-1]

    start_i = int(round(thr_start * 1000))
    stop_i  = int(round(thr_stop  * 1000))
    step_i  = int(round(thr_step  * 1000))
    if step_i <= 0:
        raise ValueError("If thr_step is too small, step_i <= 0. Please increase thr_step.")

    thr_grid_int = np.arange(start_i, stop_i, step_i, dtype=np.int16)

    min_gap = np.inf
    min_thr_int = -1
    best_acc = best_tpr = best_tnr = None

    for thr_int in thr_grid_int:

        i = np.searchsorted(scores_int_asc, thr_int, side="left")

        if i >= scores_int_asc.size:
            TP = 0.0
            FP = 0.0
        else:
            TP = float(pos_ge[i])
            FP = float(neg_ge[i])

        FN = float(W_pos - TP)
        TN = float(W_neg - FP)

        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        tnr = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        acc = (TP + TN) / (W_pos + W_neg)

        gap = abs(tpr - tnr)


        if gap < min_gap:
            min_gap = gap
            min_thr_int = int(thr_int)
            best_acc, best_tpr, best_tnr = float(acc), float(tpr), float(tnr)

    if min_thr_int < 0:
        return None

    thr_float = min_thr_int / 1000.0
    return float(thr_float), float(best_acc), float(best_tpr), float(best_tnr)


_G = {}

def _worker_init(N, pid_code_s, y_s, scores_int, group_id, thr_start, thr_stop, thr_step):
    _G["N"] = N
    _G["pid_code_s"] = pid_code_s
    _G["y_s"] = y_s
    _G["scores_int"] = scores_int
    _G["group_id"] = group_id
    _G["G"] = scores_int.size
    _G["thr_start"] = thr_start
    _G["thr_stop"] = thr_stop
    _G["thr_step"] = thr_step


def _bootstrap_chunk(n_rep, seed):
    N = _G["N"]
    pid_code_s = _G["pid_code_s"]
    y_s = _G["y_s"]
    scores_int = _G["scores_int"]
    group_id = _G["group_id"]
    G = _G["G"]

    thr_start = _G["thr_start"]
    thr_stop = _G["thr_stop"]
    thr_step = _G["thr_step"]

    rng = np.random.default_rng(seed)

    aucs, accs, tprs, tnrs, thrs = [], [], [], [], []

    for _ in range(n_rep):
        sampled = rng.integers(0, N, size=N)
        counts = np.bincount(sampled, minlength=N).astype(np.float64)

        w = counts[pid_code_s]
        if w.sum() == 0:
            continue

        wpos = w * y_s
        wneg = w - wpos

        pos = np.bincount(group_id, weights=wpos, minlength=G).astype(np.float64)
        neg = np.bincount(group_id, weights=wneg, minlength=G).astype(np.float64)

        auc = _weighted_auc_from_group(pos, neg)
        if auc is None:
            continue

        thr_acc = _threshold_scan_grid_from_group_int(
            scores_int, pos, neg,
            thr_start=thr_start, thr_stop=thr_stop, thr_step=thr_step
        )
        if thr_acc is None:
            continue

        thr, acc, tpr, tnr = thr_acc

        aucs.append(auc)
        thrs.append(thr)
        accs.append(acc)
        tprs.append(tpr)
        tnrs.append(tnr)

    return aucs, thrs, accs, tprs, tnrs


def patient_bootstrap_metrics_ci_fast_mp(
    result_id,
    y,
    p,
    B=2000,
    seed=42,
    alpha=0.05,
    n_jobs=None,
    chunk_size=200,
    thr_start=0.10,
    thr_stop=0.30,
    thr_step=0.01
):

    N, pid_code_s, y_s, scores_int, group_id = _group_prepare(result_id, y, p)
    G = scores_int.size

    w0 = np.ones_like(y_s, dtype=np.float64)
    wpos0 = w0 * y_s
    wneg0 = w0 - wpos0
    pos0 = np.bincount(group_id, weights=wpos0, minlength=G).astype(np.float64)
    neg0 = np.bincount(group_id, weights=wneg0, minlength=G).astype(np.float64)

    point_auc = _weighted_auc_from_group(pos0, neg0)
    thr0, acc0, tpr0, tnr0 = _threshold_scan_grid_from_group_int(
        scores_int, pos0, neg0,
        thr_start=thr_start, thr_stop=thr_stop, thr_step=thr_step
    )

    if n_jobs is None:
        cpu = os.cpu_count() or 4
        n_jobs = min(cpu, max(1, B // 50))
    n_jobs = max(1, int(n_jobs))

    chunks = []
    remaining = B
    while remaining > 0:
        n = min(chunk_size, remaining)
        chunks.append(n)
        remaining -= n

    ss = np.random.SeedSequence(seed)
    child_seeds = ss.spawn(len(chunks))
    child_seeds = [int(s.generate_state(1)[0]) for s in child_seeds]

    auc_boot, thr_boot, acc_boot, tpr_boot, tnr_boot = [], [], [], [], []

    with ProcessPoolExecutor(
        max_workers=n_jobs,
        initializer=_worker_init,
        initargs=(N, pid_code_s, y_s, scores_int, group_id, thr_start, thr_stop, thr_step),
    ) as ex:
        futures = [ex.submit(_bootstrap_chunk, n_rep, s) for n_rep, s in zip(chunks, child_seeds)]
        for fu in as_completed(futures):
            aucs, thrs, accs, tprs, tnrs = fu.result()
            auc_boot.extend(aucs)
            thr_boot.extend(thrs)
            acc_boot.extend(accs)
            tpr_boot.extend(tprs)
            tnr_boot.extend(tnrs)

    if len(auc_boot) == 0:
        raise ValueError("All bootstrap samples are invalid (possibly only a single class each time), making it impossible to obtain the distribution.")

    def ci(arr):
        arr = np.asarray(arr, dtype=np.float64)
        lo, hi = np.percentile(arr, [100 * (alpha / 2), 100 * (1 - alpha / 2)])
        return round(lo, 3), round(hi, 3)

    return {
        "N_patients": int(N),
        "n_valid_boot": int(len(auc_boot)),
        "n_jobs": int(n_jobs),

        "threshold_point": round(thr0, 3),
        "threshold_ci": ci(thr_boot),

        "AUC_point": round(point_auc, 3),
        "AUC_ci": ci(auc_boot),

        "ACC_point": round(acc0, 3),
        "ACC_ci": ci(acc_boot),

        "TPR_point": round(tpr0, 3),
        "TPR_ci": ci(tpr_boot),

        "TNR_point": round(tnr0, 3),
        "TNR_ci": ci(tnr_boot),
    }



if __name__ == "__main__":
    res6 = patient_bootstrap_metrics_ci_fast_mp(
        result_id, test_label_all_6, test_prob_all_6,
        B=10000, seed=42, n_jobs=48, chunk_size=250,
        thr_start=0.10, thr_stop=0.30, thr_step=0.01
    )
    res12 = patient_bootstrap_metrics_ci_fast_mp(
        result_id, test_label_all_12, test_prob_all_12,
        B=10000, seed=42, n_jobs=48, chunk_size=250,
        thr_start=0.10, thr_stop=0.30, thr_step=0.01
    )
    res24 = patient_bootstrap_metrics_ci_fast_mp(
        result_id, test_label_all_24, test_prob_all_24,
        B=10000, seed=42, n_jobs=48, chunk_size=250,
        thr_start=0.10, thr_stop=0.30, thr_step=0.01
    )

    print("6h :", res6)
    print("12h:", res12)
    print("24h:", res24)
